# Лабораторна 6. Власна RAG-система: асистент з документації Laravel 13

Система відповідає українською на питання про фреймворк Laravel за офіційною документацією й показує, з якого розділу взято відповідь.

## 1.1. Корпус

**Корпус:** офіційна документація фреймворку **Laravel**, гілка **13.x** (актуальна версія), репозиторій [laravel/docs](https://github.com/laravel/docs) на GitHub.
- **Ліцензія:** MIT (файл `license.md` у репозиторії, виводиться нижче). Корпус у репозиторій лаби не додаємо: він завантажується з GitHub однією командою.
- **Формат:** файли Markdown, англійською мовою, з прикладами коду (PHP, Blade, команди shell).
- **Обсяг:** вибрано **40 розділів**, які використовуються в розробці типового вебзастосунку й у курсі «Сучасні фреймворки вебпрограмування»: встановлення, конфігурація, маршрути, middleware, контролери, запити й відповіді, Blade, валідація, автентифікація й авторизація, база даних, міграції, сідери, Eloquent та зв'язки, пагінація, черги, тестування тощо. Точна кількість символів і фрагментів — у виводі клітинок нижче. Спеціалізовані пакети (Cashier, Horizon, Telescope, Reverb та ін.) не включено.

**Хто й навіщо ставить питання:** студенти, які виконують лабораторні з Laravel, і початківці-розробники. Типові питання: «якою командою створити міграцію», «як захистити маршрут», «як у моделі вказати назву таблиці». Відповідь потрібна **коротка, з прикладом коду й посиланням на розділ документації**, щоб її можна було перевірити.

**Чому саме RAG, а не промпт чи донавчання:**
- **Документація оновлюється з кожною версією.** У Laravel 13 змінився синтаксис моделей: замість властивостей `protected $table`, `$fillable`, `$timestamps` тепер використовуються атрибути PHP `#[Table(...)]`, `#[Fillable(...)]`, `#[WithoutTimestamps]`. Мовні моделі тренувалися на старих версіях і впевнено відповідають застарілим синтаксисом (перевірено нижче). RAG бере відповідь з **актуального** тексту, і для нової версії достатньо перебудувати індекс.
- **Промпт не вмістить документацію:** 40 розділів — це ~1.4 млн символів (сотні тисяч токенів), значно більше за контекст малої моделі.
- **Донавчання** дороге, вимагає тисяч прикладів «питання — відповідь» і не дає посилання на джерело. Після кожного оновлення документації його довелося б повторювати, і модель усе одно могла б «змішувати» старий і новий синтаксис.
- RAG дає **цитування**: користувач бачить розділ документації, з якого взято відповідь, і може його перевірити.

In [1]:
!pip install -q "sentence-transformers>=5" rank_bm25 faiss-cpu openai gradio

In [2]:
import json
import os
import re
import subprocess
import time
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder, SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32
print("Пристрій:", device, "|", torch.cuda.get_device_name() if device == "cuda" else "увімкніть GPU T4")
pd.set_option("display.max_colwidth", 120)

Пристрій: cuda | Tesla T4


In [3]:
# Завантажуємо документацію Laravel з GitHub (лише останній стан гілки за замовчуванням)
DOCS_DIR = Path("laravel-docs")
if not DOCS_DIR.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/laravel/docs", str(DOCS_DIR)], check=True)
branch = subprocess.run(["git", "-C", str(DOCS_DIR), "rev-parse", "--abbrev-ref", "HEAD"], capture_output=True, text=True).stdout.strip()
commit = subprocess.run(["git", "-C", str(DOCS_DIR), "log", "-1", "--format=%h %cs"], capture_output=True, text=True).stdout.strip()
print(f"Гілка: {branch}, коміт: {commit}")
print(f"Усього файлів документації: {len(list(DOCS_DIR.glob('*.md')))}")
print("\nЛіцензія:\n", (DOCS_DIR / "license.md").read_text()[:120], "…")

Гілка: 13.x, коміт: 2c2cdcd 2026-11-19
Усього файлів документації: 105

Ліцензія:
 The MIT License (MIT)

Copyright (c) Taylor Otwell

Permission is hereby granted, free of charge, to any person obtainin …


### Підготовка тексту й розбиття на фрагменти

- Прибираємо службову розмітку: якорі `<a name=…>`, шаблон версії в посиланнях, Markdown-посилання (лишаємо лише текст), зміст на початку сторінки.
- Ділимо кожен розділ документації **за заголовками** `##`, `###`, `####`. Розділ документації — природна одиниця змісту, і код-приклади не розриваються від свого пояснення.
- Якщо розділ довший за ліміт, ріжемо його **за абзацами** з перекриттям 150 символів.
- **На початок кожного фрагмента дописуємо шлях заголовків**, наприклад `Eloquent: Getting Started > Timestamps`. Як у демо 1 (розділ 9), це повертає фрагменту контекст: багато фрагментів (наприклад, приклад коду) без назви розділу втрачають зміст.

In [4]:
import re
from pathlib import Path

FILES = [
    "installation", "configuration", "structure", "routing", "middleware", "csrf", "controllers", "requests",
    "responses", "views", "blade", "urls", "session", "validation", "errors", "logging", "authentication",
    "authorization", "database", "queries", "pagination", "migrations", "seeding", "eloquent",
    "eloquent-relationships", "eloquent-collections", "eloquent-mutators", "eloquent-factories", "artisan",
    "filesystem", "mail", "queues", "testing", "deployment", "hashing", "passwords", "verification",
    "localization", "cache", "scheduling",
]


def clean_markdown(text):
    text = re.sub(r'<a name="[^"]*"></a>\s*', "", text)          # якорі для посилань
    text = text.replace("/docs/{{version}}/", "")                 # шаблон версії в посиланнях
    text = re.sub(r"\[([^\]]+)\]\([^)]+\)", r"\1", text)          # [текст](посилання) → текст
    text = re.sub(r"\s*\{\.[\w-]+\}", "", text)                    # службові класи на кшталт {.collection-method}
    return text


def split_sections(text):
    """Ділить markdown на розділи за заголовками ##, ###, ####; повертає (шлях заголовків, текст)."""
    title = re.search(r"^# (.+)$", text, re.M).group(1).strip()
    body = text.split("\n", 1)[1]
    # Усе до першого заголовка ## — це зміст сторінки (список посилань), його пропускаємо
    first = re.search(r"^## ", body, re.M)
    body = body[first.start():] if first else body
    sections, path, buf = [], [title], []
    in_code = False
    for line in body.split("\n"):
        if line.strip().startswith("```"):
            in_code = not in_code
        m = None if in_code else re.match(r"^(#{2,4}) (.+)$", line)
        if m:
            if "".join(buf).strip():
                sections.append((" > ".join(path), "\n".join(buf).strip()))
            level = len(m.group(1))
            path = path[: level - 1] + [m.group(2).strip()]
            buf = []
        else:
            buf.append(line)
    if "".join(buf).strip():
        sections.append((" > ".join(path), "\n".join(buf).strip()))
    return title, sections


def split_long(text, size, overlap):
    """Ріже задовгий розділ: спершу за абзацами, далі за довжиною, з перекриттям."""
    if len(text) <= size:
        return [text]
    parts, current = [], ""
    for para in re.split(r"\n\s*\n", text):
        while len(para) > size:
            parts.append(para[:size])
            para = para[size - overlap:]
        if current and len(current) + len(para) + 2 > size:
            parts.append(current)
            current = current[-overlap:] + "\n\n" + para if overlap else para
        else:
            current = f"{current}\n\n{para}" if current else para
    if current:
        parts.append(current)
    return parts


def build_chunks(docs_dir, size, overlap=150):
    chunks = []
    for name in FILES:
        raw = Path(docs_dir, f"{name}.md").read_text(encoding="utf-8")
        title, sections = split_sections(clean_markdown(raw))
        for heading, text in sections:
            for part in split_long(text, size, overlap):
                chunks.append({"doc": name, "title": title, "section": heading,
                               "text": f"{heading}\n\n{part}"})
    return chunks


# Статистика корпусу за документами
stats = []
for name in FILES:
    text = (DOCS_DIR / f"{name}.md").read_text(encoding="utf-8")
    stats.append({"файл": f"{name}.md", "заголовок": re.search(r"^# (.+)$", text, re.M).group(1), "символів": len(text)})
stats = pd.DataFrame(stats)
print(f"Документів: {len(stats)}, загальний обсяг: {stats['символів'].sum():,} символів "
      f"(~{stats['символів'].sum() / 4 / 1000:.0f} тис. токенів англійською)")
display(stats.sort_values("символів", ascending=False).head(10))

Документів: 40, загальний обсяг: 1,405,217 символів (~351 тис. токенів англійською)


,файл,заголовок,символів
31,queues.md,Queues,141338
13,validation.md,Validation,120079
24,eloquent-relationships.md,Eloquent: Relationships,98483
10,blade.md,Blade Templates,72362
23,eloquent.md,Eloquent: Getting Started,71589
21,migrations.md,Database: Migrations,61146
30,mail.md,Mail,59917
19,queries.md,Database: Query Builder,59139
16,authentication.md,Authentication,46747
3,routing.md,Routing,40714


## 1.2. Бенчмарк

Набір питань складено **до** налаштування системи. Для кожного питання вказано:
- документ(и), де є відповідь;
- **регулярний вираз-доказ**: за ним розпізнаємо фрагмент, що справді містить відповідь (перевіряти лише документ замало: у `eloquent.md` сотня фрагментів, і більшість не про те);
- **еталонну відповідь**;
- **ключ правильності**: регулярний вираз, який має бути у відповіді системи (команда, директива, метод);
- **тип питання**.

Питання сформульовані **українською** (як їх ставив би студент), а документація англійською, тож система має знаходити відповідь **між мовами**. Типи питань:
- **команда** — команди Artisan;
- **синтаксис** — як написати конкретну конструкцію (маршрути, Blade, Eloquent);
- **факт** — назва файлу, змінної, таблиці;
- **версія 13** — питання, де в Laravel 13 змінився синтаксис (атрибути PHP замість властивостей), а модель знає старий;
- **немає відповіді** — 4 питання, відповіді на які в документації немає; система має відмовитися.

Частина питань сформульована своїми словами, а не словами документації («лише залогіненим користувачам» замість «authenticated users», «багато коментарів» замість «one to many»).

In [5]:
# (питання, документи з відповіддю, регулярний вираз-доказ у фрагменті, еталонна відповідь, ключ правильної відповіді, тип)
BENCHMARK = [
    # --- команди Artisan ---
    ("Якою командою створити нову міграцію?", ["migrations"], r"php artisan make:migration",
     "php artisan make:migration create_flights_table", r"make:migration", "команда"),
    ("Як виконати всі невиконані міграції?", ["migrations"], r"php artisan migrate\b",
     "php artisan migrate", r"php artisan migrate\b", "команда"),
    ("Як відкотити останню операцію міграції?", ["migrations"], r"migrate:rollback",
     "php artisan migrate:rollback", r"migrate:rollback", "команда"),
    ("Як видалити всі таблиці й заново виконати всі міграції?", ["migrations"], r"migrate:fresh",
     "php artisan migrate:fresh", r"migrate:fresh", "команда"),
    ("Якою командою створити модель Eloquent разом із міграцією?", ["eloquent", "migrations"], r"make:model \w+ (--migration|-m)",
     "php artisan make:model Flight --migration (або -m)", r"make:model.*(--migration|-m)", "команда"),
    ("Як створити контролер за допомогою Artisan?", ["controllers"], r"make:controller",
     "php artisan make:controller UserController", r"make:controller", "команда"),
    ("Якою командою створити клас Form Request для валідації?", ["validation"], r"make:request",
     "php artisan make:request StorePostRequest", r"make:request", "команда"),
    ("Як створити новий сідер (seeder)?", ["seeding"], r"make:seeder",
     "php artisan make:seeder UserSeeder", r"make:seeder", "команда"),
    ("Як створити власний middleware?", ["middleware"], r"make:middleware",
     "php artisan make:middleware EnsureTokenIsValid", r"make:middleware", "команда"),
    # --- маршрути, контролери, шаблони ---
    ("У якому файлі описують маршрути для вебінтерфейсу?", ["routing"], r"routes/web\.php",
     "routes/web.php", r"routes/web\.php", "факт"),
    ("Як позначити параметр маршруту як необов'язковий?", ["routing"], r"\{name\?\}",
     "Додати ? після назви параметра, наприклад {name?}, і задати значення за замовчуванням змінній", r"\?\}|\{\w+\?", "синтаксис"),
    ("Як одним рядком зареєструвати всі маршрути для CRUD-контролера?", ["controllers"], r"Route::resource",
     "Route::resource('photos', PhotoController::class)", r"Route::resource", "синтаксис"),
    ("Як зробити так, щоб маршрут був доступний лише залогіненим користувачам?", ["authentication", "middleware"], r"middleware\('auth'\)",
     "Додати до маршруту ->middleware('auth')", r"middleware\(\s*'auth'", "синтаксис"),
    ("Як у шаблоні Blade вивести змінну $name з екрануванням HTML?", ["blade"], r"\{\{ \$name \}\}",
     "{{ $name }}", r"\{\{\s*\$name\s*\}\}", "синтаксис"),
    ("Як у Blade вивести дані без екранування?", ["blade"], r"\{!! \$name !!\}",
     "{!! $name !!}", r"\{!!", "синтаксис"),
    ("Яку директиву Blade додати у форму для захисту від CSRF-атак?", ["blade", "csrf"], r"@csrf",
     "@csrf", r"@csrf", "синтаксис"),
    # --- валідація ---
    ("Як провалідувати дані запиту прямо в контролері?", ["validation"], r"\$request->validate\(",
     "$request->validate([...правила...])", r"validate\(", "синтаксис"),
    ("Через яку змінну у view доступні помилки валідації?", ["validation"], r"\$errors",
     "Змінна $errors", r"\$errors", "факт"),
    # --- Eloquent (у Laravel 13 змінився синтаксис — атрибути PHP) ---
    ("Яку таблицю за замовчуванням використовує модель Flight?", ["eloquent"], r"`flights` table",
     "flights", r"flights", "факт"),
    ("Як вказати для моделі іншу назву таблиці?", ["eloquent"], r"#\[Table\('my_flights'\)\]",
     "Атрибутом #[Table('my_flights')] на класі моделі", r"#\[Table", "версія 13"),
    ("Як вимкнути автоматичне заповнення created_at і updated_at у моделі?", ["eloquent"], r"#\[Table\(timestamps: false\)\]|#\[WithoutTimestamps\]",
     "Атрибутом #[Table(timestamps: false)] або #[WithoutTimestamps]", r"timestamps:\s*false|WithoutTimestamps", "версія 13"),
    ("Як дозволити масове заповнення поля name в моделі?", ["eloquent"], r"#\[Fillable\(",
     "Атрибутом #[Fillable(['name'])] на моделі", r"Fillable", "версія 13"),
    ("Як знайти модель за id, а якщо її немає — повернути помилку 404?", ["eloquent"], r"findOrFail",
     "Метод findOrFail", r"findOrFail", "синтаксис"),
    ("Як описати зв'язок, коли в одного поста багато коментарів?", ["eloquent-relationships"], r"hasMany",
     "Метод hasMany у моделі Post: return $this->hasMany(Comment::class)", r"hasMany", "синтаксис"),
    # --- інше ---
    ("Як посторінково вивести користувачів по 15 на сторінку?", ["pagination"], r"paginate\(15\)",
     "->paginate(15)", r"paginate\(\s*15", "синтаксис"),
    ("Як захешувати пароль користувача?", ["hashing"], r"Hash::make",
     "Hash::make($password)", r"Hash::make", "синтаксис"),
    # --- немає відповіді в документації ---
    ("Як підключити Laravel до бази даних Oracle?", [], None, None, None, "немає відповіді"),
    ("Хто створив мову програмування PHP?", [], None, None, None, "немає відповіді"),
    ("Скільки коштує підписка на Laracasts?", [], None, None, None, "немає відповіді"),
    ("Яка середня зарплата Laravel-розробника в Україні?", [], None, None, None, "немає відповіді"),
]


bench_df = pd.DataFrame(BENCHMARK, columns=["питання", "документи", "доказ", "еталон", "ключ", "тип"])
print(f"Питань: {len(bench_df)}, з них без відповіді: {(bench_df['документи'].str.len() == 0).sum()}")
print(bench_df["тип"].value_counts().to_string())
display(bench_df[["питання", "документи", "еталон", "тип"]])

Питань: 30, з них без відповіді: 4
тип
синтаксис          11
команда             9
немає відповіді     4
факт                3
версія 13           3


,питання,документи,еталон,тип
0,Якою командою створити нову міграцію?,[migrations],php artisan make:migration create_flights_table,команда
1,Як виконати всі невиконані міграції?,[migrations],php artisan migrate,команда
2,Як відкотити останню операцію міграції?,[migrations],php artisan migrate:rollback,команда
3,Як видалити всі таблиці й заново виконати всі міграції?,[migrations],php artisan migrate:fresh,команда
4,Якою командою створити модель Eloquent разом із міграцією?,"[eloquent, migrations]",php artisan make:model Flight --migration (або -m),команда
5,Як створити контролер за допомогою Artisan?,[controllers],php artisan make:controller UserController,команда
6,Якою командою створити клас Form Request для валідації?,[validation],php artisan make:request StorePostRequest,команда
7,Як створити новий сідер (seeder)?,[seeding],php artisan make:seeder UserSeeder,команда
8,Як створити власний middleware?,[middleware],php artisan make:middleware EnsureTokenIsValid,команда
9,У якому файлі описують маршрути для вебінтерфейсу?,[routing],routes/web.php,факт


## 1.3. Індексація

| Компонент | Вибір | Чому |
|---|---|---|
| Модель ембедингів | `Qwen/Qwen3-Embedding-0.6B` (fp16, вектор 1024) | Багатомовна: потрібен **міжмовний** пошук (питання українською, документи англійською). Висока позиція на MTEB серед моделей такого розміру, працює на T4, перевірена в демо 1–3. Запит кодується з інструкцією `prompt_name="query"`, документи — без |
| Розбиття | за заголовками Markdown + за абзацами, 2 варіанти розміру: **800** і **1500** символів, перекриття 150 | Розділ документації — природна одиниця змісту. Два розміри порівнюємо в 1.4 |
| Контекст фрагмента | шлях заголовків на початку («Routing > Route Parameters > Optional Parameters») | Повертає фрагменту тему документа |
| Векторне сховище | **FAISS** `IndexFlatIP` (точний пошук за скалярним добутком) | Ембединги нормалізовані, тож скалярний добуток = косинусна подібність. ~4 тис. векторів — точного пошуку достатньо, приблизні індекси (HNSW, IVF) тут не потрібні |

In [6]:
EMBEDDING_MODEL = "Qwen/Qwen3-Embedding-0.6B"
embedder = SentenceTransformer(EMBEDDING_MODEL, device=device, model_kwargs={"dtype": dtype})


def build_index(chunks):
    start = time.time()
    vectors = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True,
                              batch_size=16, show_progress_bar=True).astype(np.float32)   # FAISS працює з float32
    index = faiss.IndexFlatIP(vectors.shape[1])
    index.add(vectors)
    bm25 = BM25Okapi([tokenize(c["text"]) for c in chunks])
    print(f"  {len(chunks)} фрагментів, ембединги за {time.time() - start:.0f} с")
    return {"chunks": chunks, "faiss": index, "bm25": bm25}


def tokenize(text):
    return re.findall(r"\w+", text.lower())


INDEXES = {}
for size in (800, 1500):
    chunks = build_chunks(DOCS_DIR, size=size, overlap=150)
    lengths = [len(c["text"]) for c in chunks]
    print(f"Розмір {size}: середня довжина фрагмента {np.mean(lengths):.0f} символів, максимум {max(lengths)}")
    INDEXES[size] = build_index(chunks)

example = INDEXES[800]["chunks"][1000]
print("\nПриклад фрагмента:", example["doc"], "|", example["section"])
print(example["text"][:500])

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/215 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/17.2k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.19GB            

model.safetensors: downloading bytes:           |  0.00B            

... (8 outputs left out)

Batches:   0%|          | 0/157 [00:00<?, ?it/s]

  2503 фрагментів, ембединги за 38 с
Розмір 1500: середня довжина фрагмента 798 символів, максимум 1593


Batches:   0%|          | 0/108 [00:00<?, ?it/s]

  1720 фрагментів, ембединги за 34 с

Приклад фрагмента: authentication | Authentication > Adding Custom User Providers > The Authenticatable Contract
Authentication > Adding Custom User Providers > The Authenticatable Contract

Now that we have explored each of the methods on the `UserProvider`, let's take a look at the `Authenticatable` contract. Remember, user providers should return implementations of this interface from the `retrieveById`, `retrieveByToken`, and `retrieveByCredentials` methods:

```php
<?php

namespace Illuminate\Contracts\Auth;

interface Authenticatable
{
    public function getAuthIdentifierName();
    public function 


## 1.4. Ретрив

Порівнюємо конфігурації пошуку:
1. **Семантичний пошук**, фрагменти 800 символів;
2. **Семантичний пошук**, фрагменти 1500 символів;
3. **BM25** (лексичний пошук за збігом слів);
4. **Гібридний пошук**: семантичний + BM25, об'єднані методом RRF;
5. **Гібридний + переранжування** cross-encoder `BAAI/bge-reranker-v2-m3` (багатомовний) на 20 кандидатах.

Конфігурації 3–5 будуємо на розмірі фрагментів, який виявиться кращим у 1–2.

**Метрики** (лише для 26 питань з відповіддю):
- **hit rate@k** — частка питань, для яких фрагмент-доказ є серед перших k знайдених (k = 1, 3, 5);
- **MRR** — середнє 1/r, де r — позиція першого фрагмента-доказу (0, якщо його немає серед перших 10).

In [7]:
CANDIDATES = 20
reranker = CrossEncoder("BAAI/bge-reranker-v2-m3", device=device, model_kwargs={"dtype": dtype})


def semantic(index, query, k):
    q = embedder.encode([query], normalize_embeddings=True, prompt_name="query").astype(np.float32)
    scores, ids = index["faiss"].search(q, k)
    return [(int(i), float(s)) for i, s in zip(ids[0], scores[0])]


def lexical(index, query, k):
    scores = index["bm25"].get_scores(tokenize(query))
    top = np.argsort(-scores)[:k]
    return [(int(i), float(scores[i])) for i in top]


def hybrid(index, query, k, c=60):
    fused = {}
    for ranking in (semantic(index, query, CANDIDATES), lexical(index, query, CANDIDATES)):
        for rank, (i, _) in enumerate(ranking, start=1):
            fused[i] = fused.get(i, 0) + 1 / (c + rank)
    return sorted(fused.items(), key=lambda x: -x[1])[:k]


def rerank(index, query, k):
    candidates = [i for i, _ in hybrid(index, query, CANDIDATES)]
    scores = reranker.predict([(query, index["chunks"][i]["text"]) for i in candidates])
    order = np.argsort(-scores)[:k]
    return [(candidates[j], float(scores[j])) for j in order]


def first_hit(index, results, gold, evidence):
    for rank, (i, _) in enumerate(results, start=1):
        chunk = index["chunks"][i]
        if chunk["doc"] in gold and re.search(evidence, chunk["text"]):
            return rank
    return None


def evaluate_retrieval(name, method, size):
    index = INDEXES[size]
    ranks = [first_hit(index, method(index, q, 10), gold, ev) for q, gold, ev, *_ in BENCHMARK if gold]
    row = {"конфігурація": name}
    for k in (1, 3, 5):
        row[f"hit@{k}"] = np.mean([r is not None and r <= k for r in ranks])
    row["MRR"] = np.mean([1 / r if r else 0 for r in ranks])
    return row, ranks


retrieval_rows, per_question = [], {}
for size in (800, 1500):
    row, ranks = evaluate_retrieval(f"семантичний, {size}", semantic, size)
    retrieval_rows.append(row); per_question[row["конфігурація"]] = ranks

BEST_SIZE = max((800, 1500), key=lambda s: [r for r in retrieval_rows if r["конфігурація"] == f"семантичний, {s}"][0]["MRR"])
print("Кращий розмір фрагмента за MRR:", BEST_SIZE)

for name, method in [(f"BM25, {BEST_SIZE}", lexical), (f"гібридний, {BEST_SIZE}", hybrid),
                     (f"гібридний + переранжування, {BEST_SIZE}", rerank)]:
    row, ranks = evaluate_retrieval(name, method, BEST_SIZE)
    retrieval_rows.append(row); per_question[name] = ranks

retrieval_table = pd.DataFrame(retrieval_rows).set_index("конфігурація").round(3)
display(retrieval_table)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 2.27GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.17k [00:00<?, ?B/s]

sentencepiece.bpe.model: reconstructing file:   0%|          |  0.00B / 5.07MB            

... (2 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

Кращий розмір фрагмента за MRR: 1500


,hit@1,hit@3,hit@5,MRR
конфігурація,,,,
"семантичний, 800",0.308,0.769,0.846,0.550
"семантичний, 1500",0.615,0.808,0.923,0.738
"BM25, 1500",0.192,0.231,0.231,0.216
"гібридний, 1500",0.538,0.615,0.731,0.606
"гібридний + переранжування, 1500",0.923,0.962,0.962,0.942


In [8]:
# Позиція фрагмента-доказу для кожного питання (порожньо — не знайдено серед перших 10)
answerable = [b for b in BENCHMARK if b[1]]
ranks_table = pd.DataFrame(per_question, index=[q for q, *_ in answerable])
ranks_table.insert(0, "тип", [t for *_, t in answerable])
display(ranks_table)

,тип,"семантичний, 800","семантичний, 1500","BM25, 1500","гібридний, 1500","гібридний + переранжування, 1500"
Якою командою створити нову міграцію?,команда,4.0,2.0,NaN,3.0,1.0
Як виконати всі невиконані міграції?,команда,1.0,1.0,NaN,1.0,1.0
Як відкотити останню операцію міграції?,команда,6.0,4.0,NaN,7.0,1.0
Як видалити всі таблиці й заново виконати всі міграції?,команда,3.0,1.0,NaN,1.0,1.0
Якою командою створити модель Eloquent разом із міграцією?,команда,2.0,2.0,NaN,7.0,1.0
Як створити контролер за допомогою Artisan?,команда,3.0,7.0,NaN,NaN,1.0
Якою командою створити клас Form Request для валідації?,команда,1.0,1.0,1.0,1.0,1.0
Як створити новий сідер (seeder)?,команда,1.0,1.0,1.0,1.0,1.0
Як створити власний middleware?,команда,2.0,1.0,NaN,NaN,1.0
У якому файлі описують маршрути для вебінтерфейсу?,факт,2.0,1.0,NaN,1.0,1.0


## 1.5. Генерація з цитуванням

**Модель:** `gemma3:4b` через **Ollama** з OpenAI-сумісним API (`http://localhost:11434/v1`), та сама модель і той самий спосіб підключення, що в лабораторній 4. Ollama встановлюється прямо в Colab. Gemma 3 добре пише українською й не має режиму міркування, тож відповідає швидко.

**Ретрив для генерації:** найкраща конфігурація з 1.4 (гібридний + переранжування), **k = 5** фрагментів.

**Параметри генерації:** `temperature=0` (відповідь має точно відтворювати команди й код з документації, без «творчості»), `max_tokens=400` (обов'язковий ліміт; відповідь з прикладом коду вміщується в ~150–300 токенів), контекст Ollama 8192 токени (5 фрагментів по ~1500 символів англійською ≈ 2000 токенів + інструкція + відповідь).

**Системна інструкція:** відповідати лише за фрагментами, посилатися на номер фрагмента `[n]`, код копіювати без змін, відмовлятися фіксованою фразою, якщо відповіді у фрагментах немає.

In [9]:
# Встановлюємо й запускаємо Ollama в Colab, завантажуємо модель (≈ 3 GB)
OLLAMA_MODEL = "gemma3:4b"
if subprocess.run(["which", "ollama"], capture_output=True).returncode != 0:
    subprocess.run("sudo apt-get -qq install -y zstd > /dev/null && curl -fsSL https://ollama.com/install.sh | sh",
                   shell=True, check=True)
env = dict(os.environ, OLLAMA_CONTEXT_LENGTH="8192")
ollama_server = subprocess.Popen(["ollama", "serve"], env=env, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
for _ in range(30):                      # чекаємо, поки сервер запуститься
    if subprocess.run(["ollama", "list"], capture_output=True).returncode == 0:
        break
    time.sleep(1)
subprocess.run(["ollama", "pull", OLLAMA_MODEL], check=True, stdout=subprocess.DEVNULL)
print("Ollama готова, модель:", OLLAMA_MODEL)

Ollama готова, модель: gemma3:4b


In [10]:
from openai import OpenAI

client = OpenAI(base_url="http://localhost:11434/v1", api_key="ollama")   # ключ Ollama не перевіряє
REFUSAL = "У документації немає відповіді на це питання."

SYSTEM_PROMPT = (
    "Ти — асистент, що відповідає на питання про фреймворк Laravel 13 за фрагментами офіційної документації. "
    "Фрагменти англійською, відповідай українською, коротко. "
    "Використовуй ЛИШЕ інформацію з наданих фрагментів, навіть якщо знаєш інший спосіб з інших версій Laravel. "
    "Команди й код копіюй з фрагментів без змін, у блоці коду. "
    "Після кожного твердження вказуй у квадратних дужках номер фрагмента, на якому воно ґрунтується, наприклад [2]. "
    f"Якщо у фрагментах немає відповіді на питання, відповідай лише: «{REFUSAL}»"
)


def llm(messages, max_tokens=400):
    response = client.chat.completions.create(model=OLLAMA_MODEL, messages=messages, temperature=0, max_tokens=max_tokens)
    return response.choices[0].message.content.strip()


def format_context(index, results):
    return "\n\n".join(f"[{n}] ({index['chunks'][i]['doc']}.md — {index['chunks'][i]['section']})\n{index['chunks'][i]['text']}"
                       for n, (i, _) in enumerate(results, start=1))


def rag(question, k=5, size=None):
    index = INDEXES[size or BEST_SIZE]
    results = rerank(index, question, k)
    user = f"Фрагменти документації:\n\n{format_context(index, results)}\n\nПитання: {question}"
    answer = llm([{"role": "system", "content": SYSTEM_PROMPT}, {"role": "user", "content": user}])
    sources = [index["chunks"][i] | {"score": s} for i, s in results]
    return answer, sources


answer, sources = rag("Як вказати для моделі іншу назву таблиці?")
print(answer)
for n, s in enumerate(sources, 1):
    print(f"  [{n}] {s['score']:.2f}  {s['doc']}.md — {s['section']}")

Ви можете явно вказати назву таблиці моделі за допомогою атрибуту `Table`:

```php
<?php

namespace App\Models;

use Illuminate\Database\Eloquent\Attributes\Table;
use Illuminate\Database\Eloquent\Model;

#[Table('my_flights')]
class Flight extends Model
{
    // ...
}
```
[1]
  [1] 0.94  eloquent.md — Eloquent: Getting Started > Eloquent Model Conventions > Table Names
  [2] 0.43  eloquent-relationships.md — Eloquent: Relationships > Many to Many Relationships > Table Structure > Model Structure
  [3] 0.07  eloquent-relationships.md — Eloquent: Relationships > Many to Many Relationships > Retrieving Intermediate Table Columns > Customizing the `pivot` Attribute Name
  [4] 0.04  migrations.md — Database: Migrations > Indexes > Renaming Indexes
  [5] 0.03  routing.md — Routing > Route Model Binding > Implicit Binding > Customizing the Key


### Без RAG модель відповідає застарілим синтаксисом

Ставимо питання «версії 13» тій самій моделі **без документації**.

In [11]:
for question, *_ , kind in BENCHMARK:
    if kind == "версія 13":
        plain = llm([{"role": "user", "content": question + " (Laravel). Відповідай коротко з прикладом коду."}], max_tokens=250)
        print(f"❓ {question}\n🤖 БЕЗ RAG: {plain[:400]}\n")

❓ Як вказати для моделі іншу назву таблиці?
🤖 БЕЗ RAG: Використовуйте alias у запиті:

```php
use App\Models\User;

$users = User::query()->alias('users')->get();
```

Це дозволить вам звертатися до таблиці `users` замість `users`.

❓ Як вимкнути автоматичне заповнення created_at і updated_at у моделі?
🤖 БЕЗ RAG: Щоб вимкнути автоматичне заповнення полів `created_at` та `updated_at` у моделі Laravel, додайте атрибут `timestampable` в `boot()` методі вашої моделі:

```php
namespace App\Models;

use Illuminate\Database\Eloquent\Model;

class YourModel extends Model
{
    protected $table = 'your_table'; // За потреби вкажіть назву таблиці

    protected $timestamps = false;
}
```

Змінюючи `$timestamps = fal

❓ Як дозволити масове заповнення поля name в моделі?
🤖 БЕЗ RAG: У Laravel для масового заповнення поля `name` у моделі використовуйте метод `fill()` або `setAttribute()`.

**Приклад з `fill()`:**

```php
$data = [
    'name' => 'Нове Ім'я',
];

$model = Model::create($data);
```

**

### Оцінювання на бенчмарку

Для кожного з 30 питань:
- **правильність**: для питання з відповіддю — у відповіді є **ключ** (команда / директива / метод з еталона) і це не відмова; для питання без відповіді — система відмовилася фіксованою фразою;
- **коректність цитування**: серед фрагментів, на які посилається відповідь (`[n]`), є фрагмент з правильного документа;
- **ретрив**: чи потрапив фрагмент-доказ до 5 переданих моделі.

Автоматична перевірка за ключем — проста й прозора, але не ідеальна (може не помітити правильну відповідь іншими словами або зарахувати відповідь, де ключ згадано не до місця), тож неправильні відповіді переглядаємо вручну в 1.6.

In [12]:
def cited_docs(answer, sources):
    numbers = {int(n) for n in re.findall(r"\[(\d+)\]", answer)}
    return [sources[n - 1]["doc"] for n in sorted(numbers) if 0 < n <= len(sources)]


rows = []
start = time.time()
for question, gold, evidence, reference, key, kind in BENCHMARK:
    answer, sources = rag(question)
    refused = "немає відповіді" in answer.lower()
    if gold:
        correct = (not refused) and bool(re.search(key, answer))
        citation_ok = any(d in gold for d in cited_docs(answer, sources))
        retrieved = any(s["doc"] in gold and re.search(evidence, s["text"]) for s in sources)
    else:
        correct, citation_ok, retrieved = refused, None, None
    rows.append({"питання": question, "тип": kind, "правильно": correct, "цитування": citation_ok,
                 "доказ у контексті": retrieved, "відмова": refused, "документи": gold,
                 "знайдено": [s["doc"] for s in sources], "еталон": reference, "відповідь": answer})
evaluation = pd.DataFrame(rows)
print(f"Оцінювання {len(rows)} питань за {(time.time() - start) / 60:.1f} хв\n")

has_answer = evaluation["документи"].str.len() > 0
print(f"Правильних відповідей на питання з відповіддю: {evaluation[has_answer]['правильно'].mean():.0%} "
      f"({evaluation[has_answer]['правильно'].sum()} з {has_answer.sum()})")
print(f"Коректних цитувань: {evaluation[has_answer]['цитування'].astype(bool).mean():.0%}")
print(f"Фрагмент-доказ у контексті: {evaluation[has_answer]['доказ у контексті'].astype(bool).mean():.0%}")
print(f"Правильних відмов на питання без відповіді: {evaluation[~has_answer]['правильно'].mean():.0%}")
print(f"Хибних відмов (відмовилась, хоча відповідь є): {evaluation[has_answer]['відмова'].sum()}")
print("\nПравильність за типами питань:")
print(evaluation.groupby("тип")["правильно"].mean().round(2).to_string())

Оцінювання 30 питань за 1.5 хв

Правильних відповідей на питання з відповіддю: 85% (22 з 26)
Коректних цитувань: 96%
Фрагмент-доказ у контексті: 96%
Правильних відмов на питання без відповіді: 100%
Хибних відмов (відмовилась, хоча відповідь є): 0

Правильність за типами питань:
тип
версія 13          1.00
команда            1.00
немає відповіді    1.00
синтаксис          0.64
факт               1.00


In [13]:
display(evaluation[["питання", "тип", "правильно", "цитування", "доказ у контексті", "відповідь"]])

,питання,тип,правильно,цитування,доказ у контексті,відповідь
0,Якою командою створити нову міграцію?,команда,True,True,True,Ви можете використовувати команду `php artisan make:migration` [1].
1,Як виконати всі невиконані міграції?,команда,True,True,True,Виконайте команду `php artisan migrate`. [1]
2,Як відкотити останню операцію міграції?,команда,True,True,True,"Щоб відкотити останню операцію міграції, можна використати команду `php artisan migrate:rollback` [1]. Також можна ..."
3,Як видалити всі таблиці й заново виконати всі міграції?,команда,True,True,True,"Щоб видалити всі таблиці та виконати всі міграції, використовуйте команду `php artisan migrate:fresh` [1]. Можна до..."
4,Якою командою створити модель Eloquent разом із міграцією?,команда,True,True,True,Використовуйте команду `php artisan make:model Flight --migration` [1]. Також можна використовувати короткий формат ...
5,Як створити контролер за допомогою Artisan?,команда,True,True,True,"Щоб створити контролер за допомогою Artisan, використовуйте команду:\n\n```shell\nphp artisan make:controller UserCo..."
6,Якою командою створити клас Form Request для валідації?,команда,True,True,True,Використовуйте команду `php artisan make:request StorePostRequest` [1].
7,Як створити новий сідер (seeder)?,команда,True,True,True,"Для створення нового сідер (seeder) виконайте команду `php artisan make:seeder <НазваСідеру>` [1]. Наприклад, `php a..."
8,Як створити власний middleware?,команда,True,True,True,"Щоб створити власний middleware, використовуйте команду `php artisan make:middleware EnsureTokenIsValid` [1]. Це ств..."
9,У якому файлі описують маршрути для вебінтерфейсу?,факт,True,True,True,"Фрагмент [1] вказує, що маршрути для вебінтерфейсу визначаються у файлі `routes/web.php`."


## 1.6. Аналіз відмов

Усі питання, де відповідь неправильна або цитування некоректне. Для кожного видно, чи був фрагмент-доказ серед 5 переданих моделі:
- **немає** → **промах ретриву**: правильний фрагмент не знайдено, і виправляти треба пошук;
- **є** → модель **не скористалася контекстом** або скористалася неправильно (галюцинація попри контекст, змішування версій тощо), або помилилась автоматична перевірка.

In [14]:
failures = evaluation[(~evaluation["правильно"]) | (evaluation["цитування"] == False)]
print(f"Неправильних або з некоректним цитуванням: {len(failures)} з {len(evaluation)}\n")
for _, row in failures.iterrows():
    verdict = "—" if not row["документи"] else ("доказ БУВ у контексті" if row["доказ у контексті"] else "ПРОМАХ РЕТРИВУ")
    print(f"❓ {row['питання']}  [{row['тип']}]")
    print(f"   очікувані документи: {row['документи']}, знайдено: {row['знайдено']} → {verdict}")
    print(f"   правильно: {row['правильно']}, цитування: {row['цитування']}")
    print(f"   еталон: {row['еталон']}")
    print(f"   відповідь: {row['відповідь'][:600]}\n")

Неправильних або з некоректним цитуванням: 5 з 30

❓ Як позначити параметр маршруту як необов'язковий?  [синтаксис]
   очікувані документи: ['routing'], знайдено: ['routing', 'routing', 'routing', 'routing', 'routing'] → доказ БУВ у контексті
   правильно: False, цитування: True
   еталон: Додати ? після назви параметра, наприклад {name?}, і задати значення за замовчуванням змінній
   відповідь: Використовуйте знак `?` після назви параметра в імені маршруту. [1]

❓ Як у шаблоні Blade вивести змінну $name з екрануванням HTML?  [синтаксис]
   очікувані документи: ['blade'], знайдено: ['blade', 'blade', 'blade', 'blade', 'views'] → доказ БУВ у контексті
   правильно: False, цитування: True
   еталон: {{ $name }}
   відповідь: Ви можете вивести змінну `$name` з екрануванням HTML, використовуючи синтаксис `!!`. [2]

```blade
Hello, {!! $name !!}.
```

❓ Яку директиву Blade додати у форму для захисту від CSRF-атак?  [синтаксис]
   очікувані документи: ['blade', 'csrf'], знайдено: ['blade', '

### Розбір випадків

Автоматична перевірка знайшла 5 проблемних відповідей з 30. Переглянувши їх вручну, бачимо, що лише одна з них — справжній промах ретриву, дві — помилки генерації, а ще дві — помилки самої автоматичної перевірки (відповідь фактично правильна).

| # | Питання | Що сталося | Тип відмови | Що можна змінити |
|---|---|---|---|---|
| 1 | Як позначити параметр маршруту як необов'язковий? | Модель відповіла «додайте знак ? після назви параметра» — це правильно, але без прикладу `{name?}`, тож ключ-регулярка не спрацювала. Також не згадано, що змінній треба задати значення за замовчуванням | Помилка автоматичної перевірки + неповна відповідь | Робити ключ менш суворим (шукати просто `?` поруч зі словом «параметр») або перевіряти відповіді LLM-суддею; у промпті вимагати приклад коду |
| 2 | Як у шаблоні Blade вивести змінну $name з екрануванням HTML? | Правильний фрагмент був у контексті, але модель дала `{!! $name !!}` — це вивід **без** екранування, тобто навпаки. У фрагменті [2] обидва синтаксиси стоять поруч, і мала модель їх переплутала | Галюцинація попри контекст (змішування двох схожих конструкцій) | Більша генеративна модель; менше фрагментів у контексті (k=3), щоб було менше схожого «шуму»; у промпті просити цитувати рядок коду дослівно |
| 3 | Яку директиву Blade додати у форму для захисту від CSRF-атак? | Відповідь `@csrf` правильна, але посилання записане як `[1, 2, 3, 4]`, а перевірка розпізнає лише формат `[1]`, тому цитування зараховано як некоректне | Помилка автоматичної перевірки (формат цитування) | Розпізнавати списки номерів у дужках; у промпті показати приклад формату `[1][2]` |
| 4 | Як провалідувати дані запиту прямо в контролері? | Фрагмент з `$request->validate(` не потрапив у топ-10 жодної конфігурації: пошук знаходив розділи про Form Request, де теж є слова «validation», «controller». Модель не відмовилась, а переказала нерелевантний фрагмент англійською | Промах ретриву + генерація попри відсутність відповіді | Перекладати або переформульовувати запит англійською перед пошуком; додавати до фрагментів короткий опис від LLM (contextual retrieval); посилити правило відмови в промпті |
| 5 | Як описати зв'язок, коли в одного поста багато коментарів? | Правильний фрагмент з `hasMany` був у контексті, але модель відповіла лише назвою «one-to-many relationship (Has Many)» без коду, тож ключ `hasMany` не знайдено | Помилка генерації (неповна відповідь, не використано код із контексту) | Вимагати в промпті приклад коду з фрагмента; більша модель; few-shot приклад відповіді |

**Висновок:** ретрив з переранжуванням майже не помиляється (1 промах з 26), тож основне джерело помилок — мала генеративна модель (4B), яка плутає схожі конструкції або відповідає надто коротко, а також надто сувора автоматична перевірка.

## 1.7. Демонстрація: інтерфейс Gradio

Поле для питання, кнопка й відповідь разом із джерелами (документ, розділ, оцінка переранжування й посилання на розділ на laravel.com). Після запуску клітинки з'явиться вікно інтерфейсу й публічне посилання `…gradio.live`.

In [15]:
import gradio as gr


def ask(question):
    if not question.strip():
        return "Введіть питання.", ""
    try:
        answer, sources = rag(question)
    except Exception as e:
        return f"⚠️ Помилка: {e}", ""
    lines = []
    for n, s in enumerate(sources, start=1):
        url = f"https://laravel.com/docs/{branch}/{s['doc']}"
        lines.append(f"**[{n}]** [{s['doc']}.md — {s['section']}]({url}) · оцінка {s['score']:.2f}")
    return answer, "\n\n".join(lines)


with gr.Blocks(title="Laravel RAG") as demo:
    gr.Markdown(f"# 📘 Асистент з документації Laravel {branch}\nПитайте українською: відповідь ґрунтується лише на офіційній документації.")
    question = gr.Textbox(label="Питання", placeholder="Наприклад: Як створити міграцію?")
    button = gr.Button("Запитати", variant="primary")
    answer_box = gr.Markdown(label="Відповідь")
    gr.Markdown("### Джерела")
    sources_box = gr.Markdown()
    gr.Examples(["Як створити контролер?", "Як захистити форму від CSRF?",
                 "Як вимкнути timestamps у моделі?", "Як підключити Laravel до Oracle?"], inputs=question)
    button.click(ask, inputs=question, outputs=[answer_box, sources_box])
    question.submit(ask, inputs=question, outputs=[answer_box, sources_box])

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://eac782505f4a2414f7.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


---
## 1.8. Мій звіт

* **Корпус і чому потрібен RAG:** офіційна документація Laravel 13 (github.com/laravel/docs, гілка 13.x, коміт 2c2cdcd від 2026-11-19, ліцензія MIT). Взято 40 основних розділів з 105 (маршрути, контролери, Blade, валідація, Eloquent, міграції, черги тощо) у форматі Markdown, загалом 1 405 217 символів (≈351 тис. токенів). Питання ставитимуть розробники-початківці, зокрема студенти, які пишуть лабораторні на Laravel. RAG потрібен, бо: документація не вміщується в промпт малої моделі; донавчання дороге й застаріває з кожною версією; у Laravel 13 змінився синтаксис моделей (атрибути `#[Table]`, `#[Fillable]` замість властивостей), а модель без RAG відповідає старим або взагалі вигаданим синтаксисом (див. демо без RAG: `alias()`, `protected $timestamps`, `fill()`), натомість з RAG — правильно, з посиланням на розділ документації.
* **Бенчмарк:** 30 питань українською, складені до налаштування системи: 9 про команди Artisan, 11 про синтаксис, 3 факти, 3 питання про синтаксис, що змінився в Laravel 13, і 4 питання без відповіді в документації (Oracle, автор PHP, ціна Laracasts, зарплата). Для кожного питання вказано документ, регулярний вираз для пошуку фрагмента-доказу, еталонну відповідь і ключ правильності.
* **Індексація (модель ембедингів, розбиття, сховище):** модель `Qwen/Qwen3-Embedding-0.6B` (багатомовна, бо питання українською, а документи англійською; запити кодуються з промптом `query`). Розбиття за заголовками Markdown, задовгі розділи ріжуться за абзацами з перекриттям 150 символів, на початок фрагмента дописується шлях заголовків. Два розміри: 800 символів → 2503 фрагменти (середня довжина 616), 1500 символів → 1720 фрагментів (середня 798). Сховище — FAISS `IndexFlatIP` (точний пошук, для кількох тисяч векторів наближені індекси не потрібні) + BM25 для лексичного пошуку. Ембединги рахуються ≈35–40 с на T4.
* **Результати ретриву для різних конфігурацій:** (26 питань з відповіддю)

  | Конфігурація | hit@1 | hit@3 | hit@5 | MRR |
  |---|---|---|---|---|
  | семантичний, 800 | 0.308 | 0.769 | 0.846 | 0.550 |
  | семантичний, 1500 | 0.615 | 0.808 | 0.923 | 0.738 |
  | BM25, 1500 | 0.192 | 0.231 | 0.231 | 0.216 |
  | гібридний (RRF), 1500 | 0.538 | 0.615 | 0.731 | 0.606 |
  | гібридний + переранжування, 1500 | **0.923** | **0.962** | **0.962** | **0.942** |

  Більші фрагменти (1500) кращі: розділ документації рідше розрізається, і пояснення залишається поруч із кодом. BM25 слабкий, бо питання українською, а документація англійською — збігаються лише назви команд і функцій. Через це гібридний пошук гірший за чисто семантичний: RRF домішує шум від BM25. Переранжування cross-encoder `BAAI/bge-reranker-v2-m3` на 20 кандидатах дає найбільший приріст, особливо hit@1 (0.615 → 0.923).
* **Генеративна модель і результати відповідей:** `gemma3:4b` через Ollama (OpenAI-сумісний API, як у лабораторній 4), temperature 0, max_tokens 400, контекст — 5 фрагментів після переранжування. Оцінювання 30 питань зайняло 1.5 хв. Правильних відповідей на питання з відповіддю — 85% (22 з 26), коректних цитувань — 96%, фрагмент-доказ у контексті — 96%, правильних відмов на питання без відповіді — 100% (4 з 4), хибних відмов — 0. За типами: команди, факти й «версія 13» — 100%, синтаксис — 64%.
* **Аналіз відмов:** з 5 проблемних відповідей лише 1 — промах ретриву (валідація в контролері: пошук знаходив розділи про Form Request), 2 — помилки генерації (переплутано `{{ }}` і `{!! !!}`, відповідь про `hasMany` без коду), ще 2 — помилки автоматичної перевірки (правильні відповіді, але в іншому формулюванні чи форматі цитування). Отже, реальна правильність вища за 85%.
* **Висновок:** RAG над документацією Laravel 13 працює: переранжування підняло hit@1 до 92%, а модель відповідає актуальним синтаксисом Laravel 13, з посиланням на розділ документації, і відмовляється, коли відповіді немає. Без RAG та сама модель відповідала застарілим або вигаданим кодом. Найслабше місце тепер — мала генеративна модель і сувора перевірка за ключем; покращити можна більшою моделлю, перекладом запиту англійською перед пошуком і перевіркою відповідей LLM-суддею.